### Importation des modules

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder

### Chargement des données

In [3]:
# Optionnel mais pratique pour l'affichage dans Jupyter
pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")

# Charger les informations des chiens
df_dogs = pd.read_excel(r"C:\Users\Cours APT\Documents\dog-behavior-analysis\data\DogInfo.xlsx")
print(f"Nombre de chiens : {len(df_dogs)}")
display(df_dogs.head())

# Charger les séries temporelles relatives à chaque chien / test
na_values = ['<undefined>']

df_ts = pd.read_csv(
    r"C:\Users\Cours APT\Documents\dog-behavior-analysis\data\DogMoveData.csv", 
    na_values=na_values,
    low_memory=False
)

print(f"Dimensions de l'échantillon : {df_ts.shape}")
display(df_ts.head())

Nombre de chiens : 45


,DogID,Breed,Weight,Age months,Gender,NeuteringStatus
0,16,Crossbreed,13,20,1,0
1,18,Belgian Shepherd,29,76,2,0
2,19,German Shepherd,30,57,1,0
3,20,Border Collie,15,47,1,0
4,21,Golden Retriever,39,19,2,0


Dimensions de l'échantillon : (10611068, 20)


,DogID,TestNum,t_sec,ABack_x,ABack_y,ABack_z,ANeck_x,ANeck_y,ANeck_z,GBack_x,GBack_y,GBack_z,GNeck_x,GNeck_y,GNeck_z,Task,Behavior_1,Behavior_2,Behavior_3,PointEvent
0,16,1,0.00,0.041504,0.938965,-0.015137,-0.067871,-0.510254,-0.934570,-17.639161,-22.766115,7.446290,-7.934571,6.347657,13.427735,NaN,NaN,NaN,NaN,NaN
1,16,1,0.01,0.041992,0.941895,-0.020020,-0.128906,-0.494141,-0.913086,-15.075685,-11.413575,4.821778,-3.906250,4.394532,16.540528,NaN,Synchronization,NaN,NaN,NaN
2,16,1,0.02,0.040527,0.939453,-0.004395,-0.158691,-0.480469,-0.911133,-12.207032,-0.122070,2.807617,-0.488281,-1.953125,26.794435,NaN,Synchronization,NaN,NaN,NaN
3,16,1,0.03,0.021484,0.946289,0.007813,-0.122070,-0.486816,-0.880371,-9.460450,7.995606,1.586914,1.159668,-5.676270,38.085940,NaN,Synchronization,NaN,NaN,NaN
4,16,1,0.04,-0.000977,0.951172,0.033691,-0.053711,-0.500000,-0.807129,-8.361817,14.587403,-1.037598,4.577637,4.089356,41.503909,NaN,Synchronization,NaN,NaN,NaN


### Feature engineering : 
- on retire les comportements de "synchronization".
- on créé une nouvelle variable sur la 'taille' du chien (petit, moyen, grand) en fonction de son poids.
- on a déjà retiré les portions de séries temporelles avec valeur de Behavior_1 manquante = pour le train uniquement !.

In [4]:
# 1. Définition des seuils de poids (à ajuster selon ta connaissance vétérinaire si besoin)
# Exemple : < 15 kg (Petit), 15 à 30 kg (Moyen), > 30 kg (Grand)
bins = [0, 15, 30, 100] 
labels = ['Taille_Petite', 'Taille_Moyenne', 'Taille_Grande']

# 2. Création de la catégorie textuelle
df_dogs['Taille_Categorie'] = pd.cut(df_dogs['Weight'], bins=bins, labels=labels, right=False)

# 3. Création des colonnes binaires (0 ou 1) via One-Hot Encoding
df_dogs = pd.get_dummies(df_dogs, columns=['Taille_Categorie'], dtype=int)

# Affichage pour vérifier les nouvelles colonnes
display(df_dogs[['DogID', 'Breed', 'Weight', 'Taille_Categorie_Taille_Petite', 'Taille_Categorie_Taille_Moyenne', 'Taille_Categorie_Taille_Grande']].head())

# Liste des comportements techniques à exclure
comportements_a_ignorer = ['Synchronization', 'Extra_Synchronization']

# Filtrage pour ne conserver que les vrais comportements
df_ts = df_ts[~df_ts['Behavior_1'].isin(comportements_a_ignorer)].copy()

# Vérification rapide
print("Comportements conservés pour la modélisation :")
print(df_ts['Behavior_1'].dropna().unique())

,DogID,Breed,Weight,Taille_Categorie_Taille_Petite,Taille_Categorie_Taille_Moyenne,Taille_Categorie_Taille_Grande
0,16,Crossbreed,13,1,0,0
1,18,Belgian Shepherd,29,0,1,0
2,19,German Shepherd,30,0,0,1
3,20,Border Collie,15,0,1,0
4,21,Golden Retriever,39,0,0,1


Comportements conservés pour la modélisation :
<ArrowStringArray>
[        'Walking',         'Shaking',        'Sniffing',          'Eating',
         'Sitting',        'Trotting',          'Pacing',     'Lying chest',
         'Playing',        'Standing',         'Panting',        'Drinking',
       'Galloping', 'Carrying object',         'Tugging',         'Jumping',
          'Bowing']
Length: 17, dtype: str


### Séparation des jeux en train / test + vérification de l'équilibre des classes

In [5]:
# 1. On exclut les lignes non labellisées pour se concentrer sur les données utiles
df_annotated = df_ts.dropna(subset=['Behavior_1']).copy()

# 2. Définition du split 80/20 basé sur les groupes (DogID)
# Le random_state garantit que tu obtiendras toujours le même split en relançant la cellule
gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=2)

# Récupération des indices pour le Train et le Test
train_idx, test_idx = next(gss.split(df_annotated, groups=df_annotated['DogID']))

df_train = df_annotated.iloc[train_idx]
df_test = df_annotated.iloc[test_idx]

# Isoler les ID des chiens présents dans chaque ensemble
train_ids = df_train['DogID'].unique()
test_ids = df_test['DogID'].unique()

# 2. Récupérer leurs caractéristiques depuis le dataset statique
df_dogs_train = df_dogs[df_dogs['DogID'].isin(train_ids)].copy()
df_dogs_test = df_dogs[df_dogs['DogID'].isin(test_ids)].copy()

# Ajouter une colonne pour identifier la provenance
df_dogs_train['Set'] = 'Train'
df_dogs_test['Set'] = 'Test'

### Feature engineering sur les séries temporelles et création de jeux de données finaux d'apprentissage et de test
- création de nouvelles variables issues des séries temporelles

In [ ]:
def extraire_features_dynamiques(df_continu, window_size=100, step_size=50):
    """
    Extrait les features sur un enregistrement continu, puis filtre les fenêtres non labellisées.
    window_size = 100 lignes (ex: 1 seconde à 100Hz)
    step_size = 50 lignes (chevauchement de 50%)
    """
    df_continu = df_continu.copy() # Sécurité pour éviter les warnings de Pandas
    
    # 1. Calcul automatisé des features dynamiques pour les 4 capteurs
    capteurs = ['ABack', 'ANeck', 'GBack', 'GNeck']
    
    for capteur in capteurs:
        # Norme géométrique (Intensité globale du mouvement ou de la rotation)
        df_continu[f'{capteur}_norm'] = np.sqrt(
            df_continu[f'{capteur}_x']**2 + 
            df_continu[f'{capteur}_y']**2 + 
            df_continu[f'{capteur}_z']**2
        )
        
        # Jerk (Saccade : différence d'un instant à l'autre)
        df_continu[f'{capteur}_jerk'] = df_continu[f'{capteur}_norm'].diff().fillna(0)

    features_list = []
    
    # 2. Fenêtrage glissant
    for i in range(0, len(df_continu) - window_size, step_size):
        window = df_continu.iloc[i : i + window_size]
        
        # 3. Gestion des valeurs manquantes au niveau de la fenêtre
        labels_in_window = window['Behavior_1'].dropna()
        
        if len(labels_in_window) < (window_size / 2):
            continue
            
        label_majoritaire = labels_in_window.mode().iloc[0]
        
        # 4. Initialisation du dictionnaire de la fenêtre
        window_features = {
            'DogID': window['DogID'].iloc[0],
            'TestNum': window['TestNum'].iloc[0],
            'Behavior': label_majoritaire,
            
            # Indices d'indépendance Cou/Dos (Accélération et Gyroscope)
            'Ratio_A_Neck_Back': window['ANeck_norm'].mean() / (window['ABack_norm'].mean() + 1e-6),
            'Ratio_G_Neck_Back': window['GNeck_norm'].mean() / (window['GBack_norm'].mean() + 1e-6)
        }
        
        # 5. Boucle pour extraire Moyenne et Ecart-Type de toutes les dimensions
        axes_et_metrics = ['x', 'y', 'z', 'norm', 'jerk']
        
        for capteur in capteurs:
            for axe in axes_et_metrics:
                col_name = f'{capteur}_{axe}'
                window_features[f'{col_name}_mean'] = window[col_name].mean()
                window_features[f'{col_name}_std'] = window[col_name].std()
        
        features_list.append(window_features)
        
    return pd.DataFrame(features_list)

# ==========================================
# APPLICATION ET AFFICHAGE
# ==========================================

# 1. Application de l'extraction
list_features = []
for (dog_id, test_num), group in df_ts.groupby(['DogID', 'TestNum']):
    if len(group) > 100:
        df_f = extraire_features_dynamiques(group, window_size=100, step_size=50)
        list_features.append(df_f)

df_all_features = pd.concat(list_features, ignore_index=True)
print(f"Extraction terminée : {len(df_all_features)} fenêtres générées au total.\n")
print(f"Nombre de variables (features) par fenêtre : {df_all_features.shape[1]}")

# Sauvegarde des features extraites pour ne plus jamais avoir à refaire le calcul
df_all_features.to_csv(r"C:\Users\Cours APT\Documents\dog-behavior-analysis\data\features_canines_completes.csv", index=False)

# (Pour la recharger demain en 1 seconde, tu feras juste :)
# df_all_features = pd.read_csv('features_canines_completes.csv')

# 2. Sélection des comportements (Top 10)
top_behaviors = df_all_features['Behavior'].value_counts().nlargest(10).index
df_plot = df_all_features[df_all_features['Behavior'].isin(top_behaviors)]

# 3. Choix de 4 nouvelles features hyper intéressantes à tracer
features_a_tracer = [
    'ANeck_jerk_mean',      # Repère bien les mouvements de tête secs (manger/boire/renifler)
    'GBack_norm_mean',      # Intensité de la rotation du corps (tourner en rond)
    'Ratio_A_Neck_Back',    # Différence globale de mouvement Cou vs Dos
    'Ratio_G_Neck_Back'     # Différence de rotation Cou vs Dos
]

# 4. Grille de Boxplots
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, feature in enumerate(features_a_tracer):
    sns.boxplot(
        data=df_plot, 
        x='Behavior', 
        y=feature, 
        hue='Behavior', 
        legend=False,
        ax=axes[i], 
        palette='Set2'
    )
    axes[i].set_title(f"Distribution de : {feature}", fontsize=13)
    axes[i].set_xlabel("")
    axes[i].set_ylabel("Valeur")
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### Entrainement de modèles pour l'approche 1

#### Préparation et jointure des données

In [8]:
# ==========================================
# CELLULE 1 : Préparation et Jointure des données
# ==========================================

# 1. Recharger les features sauvegardées (si besoin)
df_all_features = pd.read_csv(r"C:\Users\Cours APT\Documents\dog-behavior-analysis\data\features_canines_completes.csv")

# 2. Jointure avec les données statiques (df_dogs)
# On s'assure d'avoir bien créé nos variables catégorielles (tailles, etc.) dans df_dogs au préalable
df_combined = pd.merge(df_all_features, df_dogs, on='DogID', how='left')

# 3. Encodage de la variable cible (Behavior)
# XGBoost exige des labels numériques (0, 1, 2...)
le = LabelEncoder()
df_combined['Behavior_encoded'] = le.fit_transform(df_combined['Behavior'])
classes_names = le.classes_

# 4. Séparation stricte Train / Test basée sur les IDs définis précédemment
# (Remplace train_ids et test_ids par tes listes réelles si elles ne sont plus en mémoire)
df_train_ml = df_combined[df_combined['DogID'].isin(train_ids)].copy()
df_test_ml = df_combined[df_combined['DogID'].isin(test_ids)].copy()

# 5. Définition des blocs de features
# Toutes les colonnes qui se terminent par mean, std, ou qui contiennent Ratio ou jerk
features_baseline = [col for col in df_all_features.columns if any(x in col for x in ['_mean', '_std', 'Ratio', '_jerk'])]

# Ajout des métadonnées statiques (à adapter selon le nom exact de tes colonnes)
features_statiques = ['Weight', 'Age months', 'Gender', 'NeuteringStatus', 'Taille_Categorie_Taille_Petite', 'Taille_Categorie_Taille_Moyenne', 'Taille_Categorie_Taille_Grande']
features_augmentees = features_baseline + features_statiques

#### Configuration de la Validation Croisée et Modèles

In [ ]:
# ==========================================
# CELLULE 2 : Configuration de la Validation Croisée et Modèles
# ==========================================
from sklearn.model_selection import GroupKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, accuracy_score

# Stratégie de validation croisée groupée par chien
gkf = GroupKFold(n_splits=5)

# Dictionnaire des modèles et de leurs grilles d'hyperparamètres (volontairement restreintes pour le temps de calcul)
modeles_config = {
    'LogisticRegression': {
        'model': Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))]),
        'params': {'clf__C': [0.1, 1, 10]}
    },
    'RandomForest': {
        'model': RandomForestClassifier(class_weight='balanced', random_state=42),
        'params': {'n_estimators': [100, 200], 'max_depth': [None, 10, 20]}
    },
    # 'SVM': {
    #   'model': Pipeline([('scaler', StandardScaler()), ('clf', SVC(class_weight='balanced', probability=True))]),
    #   'params': {'clf__C': [1, 10], 'clf__kernel': ['rbf', 'linear']}
    #},
    'XGBoost': {
        'model': XGBClassifier(eval_metric='mlogloss', random_state=42),
        'params': {'n_estimators': [100, 200], 'max_depth': [3, 6], 'learning_rate': [0.01, 0.1]}
    }
}

def evaluer_modeles(features_list, nom_experience):
    print(f"\n{'='*50}\nLANCEMENT EXPÉRIENCE : {nom_experience}\n{'='*50}")
    
    X_train = df_train_ml[features_list]
    y_train = df_train_ml['Behavior_encoded']
    groups_train = df_train_ml['DogID']
    
    X_test = df_test_ml[features_list]
    y_test = df_test_ml['Behavior_encoded']
    
    resultats = {}
    
    for nom, config in modeles_config.items():
        print(f"-> Entraînement de {nom}...")
        
        # GridSearch avec GroupKFold
        grid = GridSearchCV(
            estimator=config['model'],
            param_grid=config['params'],
            cv=gkf,
            scoring='accuracy',
            n_jobs=-1 # Utilise tous les coeurs du processeur
        )
        
        # Fit en passant les groupes
        grid.fit(X_train, y_train, groups=groups_train)
        
        # Prédiction sur le Test set sanctuaire
        best_model = grid.best_estimator_
        y_pred = best_model.predict(X_test)
        
        acc = accuracy_score(y_test, y_pred)
        print(f"   Meilleurs params : {grid.best_params_}")
        print(f"   Précision (Accuracy) sur le Test : {acc:.4f}\n")
        
        resultats[nom] = {
            'best_model': best_model,
            'accuracy': acc,
            'predictions': y_pred
        }
        
    return resultats

#### Exécution Baseline vs Augmentée

In [ ]:
# ==========================================
# CELLULE 3 : Exécution Baseline vs Augmentée
# ==========================================

# 1. Entraînement sur les features dynamiques uniquement
resultats_baseline = evaluer_modeles(features_baseline, "BASELINE (Features Temporelles Uniquement)")

# 2. Entraînement avec les variables statiques ajoutées
resultats_augmentes = evaluer_modeles(features_augmentees, "AUGMENTÉE (Temporelles + Profil du Chien)")

# (Optionnel) Afficher le rapport détaillé du meilleur modèle de l'approche augmentée
meilleur_nom = max(resultats_augmentes, key=lambda k: resultats_augmentes[k]['accuracy'])
meilleur_modele = resultats_augmentes[meilleur_nom]['best_model']
meilleures_preds = resultats_augmentes[meilleur_nom]['predictions']

print(f"\n--- Rapport de classification détaillé pour le grand gagnant : {meilleur_nom} (Augmenté) ---")
print(classification_report(df_test_ml['Behavior_encoded'], meilleures_preds, target_names=classes_names))

#### Frises Chronologiques Interactives (Plotly)

In [ ]:
# ==========================================
# CELLULE 4 : Frises Chronologiques Interactives (Plotly)
# ==========================================
import plotly.graph_objects as go

def tracer_frise_chronologique(dog_id, test_num, modele, features_list):
    """
    Trace une frise interactive pour un chien donné du jeu de test.
    """
    # Isoler les données du chien
    mask = (df_test_ml['DogID'] == dog_id) & (df_test_ml['TestNum'] == test_num)
    df_plot = df_test_ml[mask].copy()
    
    if df_plot.empty:
        print(f"Aucune donnée trouvée pour DogID {dog_id} - TestNum {test_num}")
        return
        
    # Créer un axe temporel fictif basé sur l'ordre des fenêtres
    df_plot['Time_Window'] = range(len(df_plot))
    
    # Prédire avec le modèle choisi
    X_plot = df_plot[features_list]
    df_plot['Predicted_Encoded'] = modele.predict(X_plot)
    
    # Décoder pour l'affichage
    df_plot['Predicted_Behavior'] = le.inverse_transform(df_plot['Predicted_Encoded'])
    
    # Création de la figure Plotly
    fig = go.Figure()

    # Trace 1 : Vrai comportement (Vérité terrain en rouge)
    fig.add_trace(go.Scatter(
        x=df_plot['Time_Window'], 
        y=df_plot['Behavior'],
        mode='markers',
        name='Réel (Annotations)',
        marker=dict(color='red', size=10, symbol='circle', line=dict(width=1, color='darkred'))
    ))

    # Trace 2 : Comportement prédit (en vert)
    fig.add_trace(go.Scatter(
        x=df_plot['Time_Window'], 
        y=df_plot['Predicted_Behavior'],
        mode='lines+markers',
        name='Prédit (Modèle)',
        line=dict(color='green', width=2, dash='dot'),
        marker=dict(color='lightgreen', size=8, symbol='x')
    ))

    fig.update_layout(
        title=f"Frise Chronologique : Réel vs Prédit | Chien {dog_id} (Test {test_num})",
        xaxis_title="Numéro de fenêtre temporelle (Temps t)",
        yaxis_title="Comportement",
        height=600,
        hovermode="x unified",
        template="plotly_white"
    )
    
    fig.show()

# Exemple d'utilisation sur le premier chien du set de test
# On utilise le meilleur modèle de l'approche augmentée
premier_chien_test = df_test_ml['DogID'].iloc[0]
premier_test_num = df_test_ml[df_test_ml['DogID'] == premier_chien_test]['TestNum'].iloc[0]

tracer_frise_chronologique(
    dog_id=premier_chien_test, 
    test_num=premier_test_num, 
    modele=meilleur_modele, 
    features_list=features_augmentees
)